<a href="https://colab.research.google.com/github/Narmadha-H/MLA0202-FML/blob/main/ML18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---------- Common imports and helper functions ----------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")

from sklearn.datasets import load_iris, load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Perceptron as SkPerceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, roc_curve, auc,
                             r2_score, mean_squared_error, mean_absolute_error,
                             precision_score, recall_score, f1_score)

sns.set_style("whitegrid")
rng = np.random.default_rng(42)

def plot_cm(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
    plt.tight_layout(); plt.show()

def plot_reg(y_true, y_pred, title):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].scatter(y_true, y_pred, alpha=0.5, color="steelblue")
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax[0].plot(lims, lims, "r--", label="Perfect prediction")
    ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted"); ax[0].set_title(title); ax[0].legend()
    ax[1].hist(y_true - y_pred, bins=30, color="orange", edgecolor="black")
    ax[1].set_title("Residuals (Actual - Predicted)"); ax[1].set_xlabel("Error")
    plt.tight_layout(); plt.show()
    print(f"R2 = {r2_score(y_true, y_pred):.3f} | RMSE = {np.sqrt(mean_squared_error(y_true, y_pred)):.3f} | MAE = {mean_absolute_error(y_true, y_pred):.3f}")

def plot_importance(names, values, title):
    s = pd.Series(values, index=names).sort_values()
    s.plot(kind="barh", figsize=(7, 4), color="teal"); plt.title(title); plt.xlabel("Importance")
    plt.tight_layout(); plt.show()

def plot_boundary(model, X2, y, xlabel, ylabel, class_names, title):
    # model must already be trained on the 2 features in X2
    x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
    y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap="viridis")
    for i, c in enumerate(class_names):
        plt.scatter(X2[y == i, 0], X2[y == i, 1], label=c, edgecolor="k")
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title); plt.legend(); plt.show()
print("Setup complete")

class MyPerceptron:
    def __init__(self, lr=0.1, epochs=15):
        self.lr, self.epochs = lr, epochs
    def predict(self, X):
        return np.where(X @ self.w + self.b >= 0, 1, 0)
    def fit(self, X, y):
        self.w, self.b, self.errors = np.zeros(X.shape[1]), 0.0, []
        for _ in range(self.epochs):
            err = 0
            for xi, target in zip(X, y):
                update = self.lr * (target - self.predict(xi))
                self.w += update * xi; self.b += update; err += int(update != 0)
            self.errors.append(err)
        return self

iris = load_iris(); X, y = iris.data, iris.target; names = iris.target_names

# ---- Part A: two classes, two features ----
mask = y < 2
Xb = X[mask][:, [0, 2]]; yb = y[mask]               # sepal length, petal length
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=42)
p = MyPerceptron(lr=0.1, epochs=15).fit(Xb_tr, yb_tr)
print("From-scratch Perceptron test accuracy:", accuracy_score(yb_te, p.predict(Xb_te)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(range(1, len(p.errors) + 1), p.errors, marker="o"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Misclassifications")
ax[0].set_title("Perceptron Training Errors")
for i in (0, 1):
    ax[1].scatter(Xb[yb == i, 0], Xb[yb == i, 1], label=names[i], edgecolor="k")
xs = np.linspace(Xb[:, 0].min(), Xb[:, 0].max(), 50)
ax[1].plot(xs, -(p.w[0] * xs + p.b) / p.w[1], "r-", label="Decision boundary")
ax[1].set_ylim(Xb[:, 1].min() - 0.5, Xb[:, 1].max() + 0.5)
ax[1].set_xlabel("Sepal length"); ax[1].set_ylabel("Petal length"); ax[1].set_title("Setosa vs Versicolor"); ax[1].legend()
plt.tight_layout(); plt.show()

# ---- Part B: sklearn Perceptron, all 3 classes ----
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
clf = make_pipeline(StandardScaler(), SkPerceptron(max_iter=1000, eta0=0.1, random_state=42)).fit(X_tr, y_tr)
pred = clf.predict(X_te)
print("sklearn Perceptron accuracy (3 classes):", round(accuracy_score(y_te, pred), 3))
plot_cm(y_te, pred, names, "Perceptron - Confusion Matrix")

clf2 = make_pipeline(StandardScaler(), SkPerceptron(max_iter=1000, random_state=42)).fit(X[:, 2:4], y)
plot_boundary(clf2, X[:, 2:4], y, "Petal length (cm)", "Petal width (cm)", names, "Perceptron Decision Regions")